# PlacePatch — ComfyUI on Colab
Run cells top to bottom. **Runtime → Change runtime type → GPU (T4)** first.
The last cell prints a public URL: put it in `COMFYUI_BASE_URL`. Treat the URL as a secret and stop the runtime after the demo.
The URL changes every time the runtime restarts.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

In [ ]:
# Model is cached on Drive so later sessions skip the big download
import os
MODEL_URL = "https://huggingface.co/sd-legacy/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt"
CACHE = "/content/drive/MyDrive/placepatch-models"
os.makedirs(CACHE, exist_ok=True)
CK = f"{CACHE}/sd-v1-5-inpainting.ckpt"
if not os.path.exists(CK):
    !wget -q --show-progress -O "{CK}" "{MODEL_URL}"
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
!ln -sf "{CK}" /content/ComfyUI/models/checkpoints/sd-v1-5-inpainting.ckpt
!ls -lh /content/ComfyUI/models/checkpoints

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, re, requests
comfy = subprocess.Popen(["python","main.py","--port","8188","--listen","127.0.0.1"], cwd="/content/ComfyUI",
                         stdout=open("/content/comfy.log","w"), stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        if requests.get("http://127.0.0.1:8188/system_stats", timeout=2).ok: break
    except Exception: pass
    time.sleep(2)
else:
    raise RuntimeError("ComfyUI did not start; see /content/comfy.log")

tun = subprocess.Popen(["cloudflared","tunnel","--url","http://127.0.0.1:8188","--no-autoupdate"],
                       stdout=open("/content/tunnel.log","w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: url = m.group(0); break
    time.sleep(1)
print("COMFYUI_BASE_URL =", url)
print("Open that URL in a browser to use the ComfyUI UI (to build / test the workflow).")

In [ ]:
# Optional keep-alive: keep this cell running during the demo and keep the tab open
while True:
    time.sleep(60)